# 02 — Build Fact Tables

Build fact tables for ad exposures and conversions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

sns.set_theme(style='whitegrid')
%matplotlib inline

In [2]:
# Load Raw Data
RAW = '../data/raw/'

raw_ad_events = pd.read_csv(RAW + 'raw_ad_events.csv')
print({f'raw_ad_events: {raw_ad_events.shape}'})


{'raw_ad_events: (341675, 11)'}


In [3]:
# ==========================================
# BUILD FACT TABLE: fact_ad_exposures
# ==========================================

# Ensure timestamp is datetime
raw_ad_events["timestamp"] = pd.to_datetime(raw_ad_events["timestamp"])


# Create user-level ad exposure table
fact_ad_exposures = (
    raw_ad_events 
    .groupby('user_id')
    .agg(
        total_ad_events = ('event_id', 'count'),
        total_impressions = ('impression_flag', 'sum'),
        total_clicks = ('click_flag', 'sum'),
        total_video_starts = ('video_start_flag', 'sum'),
        total_video_completes = ('video_complete_flag', 'sum'),
        first_exposure = ('timestamp', 'min'),
        last_exposure_date = ('timestamp', 'max'),
        platforms_reached = ('platform', 'unique'),
        campaigns_reached = ('campaign_id', 'unique'),
        creatives_seen = ('creative_id', 'unique')
    )
    .reset_index()
)

# Days between first and last exposure
fact_ad_exposures['days_exposed'] = (
    fact_ad_exposures['last_exposure_date'] -
    fact_ad_exposures['first_exposure']
).dt.days + 1

# Exposure Flag
fact_ad_exposures['exposed_flag'] = 1

display(fact_ad_exposures.head())
print(fact_ad_exposures.shape)

,user_id,total_ad_events,total_impressions,total_clicks,total_video_starts,total_video_completes,first_exposure,last_exposure_date,platforms_reached,campaigns_reached,creatives_seen,days_exposed,exposed_flag
0,U1000001,9,9,2,6,3,2024-04-02 04:33:40,2024-04-27 22:06:08,"[TikTok, Programmatic Display, Meta, YouTube, ...","[CMP009, CMP005, CMP001, CMP007, CMP004, CMP006]","[CR008, CR019, CR004, CR023, CR012, CR002, CR0...",26,1
1,U1000002,66,66,0,54,26,2024-04-01 20:59:38,2024-04-28 19:51:52,"[YouTube, TikTok, Programmatic Display, Meta, ...","[CMP007, CMP002, CMP005, CMP009, CMP001, CMP00...","[CR013, CR006, CR022, CR005, CR003, CR008, CR0...",27,1
2,U1000004,19,19,0,13,8,2024-04-01 18:14:56,2024-04-28 20:34:23,"[CTV, Meta, TikTok, YouTube, Programmatic Disp...","[CMP004, CMP006, CMP002, CMP009, CMP003, CMP00...","[CR014, CR023, CR005, CR006, CR012, CR007, CR0...",28,1
3,U1000007,19,19,0,14,7,2024-04-01 09:46:54,2024-04-27 23:30:40,"[Meta, TikTok, CTV, Programmatic Display, YouT...","[CMP001, CMP006, CMP002, CMP008, CMP005, CMP00...","[CR003, CR023, CR007, CR008, CR014, CR022, CR0...",27,1
4,U1000009,76,76,1,58,29,2024-04-02 02:08:37,2024-04-29 00:48:39,"[TikTok, YouTube, Meta, CTV, Programmatic Disp...","[CMP002, CMP003, CMP001, CMP004, CMP007, CMP00...","[CR007, CR009, CR003, CR013, CR018, CR011, CR0...",27,1


(9718, 13)


In [4]:
# ==========================================
# PLATFORM-LEVEL IMPRESSIONS
# ==========================================

platform_impressions = (
    raw_ad_events
    .pivot_table(
        index ='user_id',
        columns ='platform',
        values = 'impression_flag',
        aggfunc = 'sum',
        fill_value = 0 
    )
    .reset_index()
)

# Clean column names
platform_impressions.columns = [
    'user_id' if col == 'user_id' else
    col.lower()
       .replace(' ', '_')
       .replace('/', '_')
       .replace('-', '_') + '_impressions'
    for col in platform_impressions.columns
]

display(platform_impressions.head())

,user_id,ctv_impressions,meta_impressions,programmatic_display_impressions,tiktok_impressions,youtube_impressions
0,U1000001,1,4,1,1,2
1,U1000002,18,16,4,17,11
2,U1000004,4,5,2,5,3
3,U1000007,5,7,2,3,2
4,U1000009,17,19,5,11,24


In [5]:
# Join platform impressions back to fact_ad_exposures
fact_ad_exposures = fact_ad_exposures.merge(
    platform_impressions,
    on ='user_id',
    how = 'left'
)

display(fact_ad_exposures.head())
print(fact_ad_exposures.shape)

,user_id,total_ad_events,total_impressions,total_clicks,total_video_starts,total_video_completes,first_exposure,last_exposure_date,platforms_reached,campaigns_reached,creatives_seen,days_exposed,exposed_flag,ctv_impressions,meta_impressions,programmatic_display_impressions,tiktok_impressions,youtube_impressions
0,U1000001,9,9,2,6,3,2024-04-02 04:33:40,2024-04-27 22:06:08,"[TikTok, Programmatic Display, Meta, YouTube, ...","[CMP009, CMP005, CMP001, CMP007, CMP004, CMP006]","[CR008, CR019, CR004, CR023, CR012, CR002, CR0...",26,1,1,4,1,1,2
1,U1000002,66,66,0,54,26,2024-04-01 20:59:38,2024-04-28 19:51:52,"[YouTube, TikTok, Programmatic Display, Meta, ...","[CMP007, CMP002, CMP005, CMP009, CMP001, CMP00...","[CR013, CR006, CR022, CR005, CR003, CR008, CR0...",27,1,18,16,4,17,11
2,U1000004,19,19,0,13,8,2024-04-01 18:14:56,2024-04-28 20:34:23,"[CTV, Meta, TikTok, YouTube, Programmatic Disp...","[CMP004, CMP006, CMP002, CMP009, CMP003, CMP00...","[CR014, CR023, CR005, CR006, CR012, CR007, CR0...",28,1,4,5,2,5,3
3,U1000007,19,19,0,14,7,2024-04-01 09:46:54,2024-04-27 23:30:40,"[Meta, TikTok, CTV, Programmatic Display, YouT...","[CMP001, CMP006, CMP002, CMP008, CMP005, CMP00...","[CR003, CR023, CR007, CR008, CR014, CR022, CR0...",27,1,5,7,2,3,2
4,U1000009,76,76,1,58,29,2024-04-02 02:08:37,2024-04-29 00:48:39,"[TikTok, YouTube, Meta, CTV, Programmatic Disp...","[CMP002, CMP003, CMP001, CMP004, CMP007, CMP00...","[CR007, CR009, CR003, CR013, CR018, CR011, CR0...",27,1,17,19,5,11,24


(9718, 18)


In [6]:
fact_ad_exposures = fact_ad_exposures.drop(
    columns=[
        "platforms_reached",
        "campaigns_reached",
        "creatives_seen"
    ]
)


In [7]:
display(fact_ad_exposures.head())
print(fact_ad_exposures.shape)

,user_id,total_ad_events,total_impressions,total_clicks,total_video_starts,total_video_completes,first_exposure,last_exposure_date,days_exposed,exposed_flag,ctv_impressions,meta_impressions,programmatic_display_impressions,tiktok_impressions,youtube_impressions
0,U1000001,9,9,2,6,3,2024-04-02 04:33:40,2024-04-27 22:06:08,26,1,1,4,1,1,2
1,U1000002,66,66,0,54,26,2024-04-01 20:59:38,2024-04-28 19:51:52,27,1,18,16,4,17,11
2,U1000004,19,19,0,13,8,2024-04-01 18:14:56,2024-04-28 20:34:23,28,1,4,5,2,5,3
3,U1000007,19,19,0,14,7,2024-04-01 09:46:54,2024-04-27 23:30:40,27,1,5,7,2,3,2
4,U1000009,76,76,1,58,29,2024-04-02 02:08:37,2024-04-29 00:48:39,27,1,17,19,5,11,24


(9718, 15)


In [8]:
# ==========================================
# FREQUENCY BUCKETS
# ==========================================

fact_ad_exposures['frequency_bucket'] = pd.cut(
    fact_ad_exposures['total_impressions'],
    bins = [0,10,20,40,60,80,100, float('inf')],
    labels = [
        '1-10',
        '11-20',
        '21-40',
        '41-60',
        '61-80',
        '81-100',
        '100+'  
    ],
    include_lowest=True
)

freq_check = (
    fact_ad_exposures['frequency_bucket']
    .value_counts()
    .sort_index()
    .to_frame('users')
)

freq_check['pct'] = (
    freq_check['users'] / freq_check['users'].sum() * 100
).round(2)

display(freq_check)

,users,pct
frequency_bucket,,
1-10,1416,14.57
11-20,1938,19.94
21-40,3371,34.69
41-60,1948,20.05
61-80,706,7.26
81-100,219,2.25
100+,120,1.23


### Frequency Distribution

Exposure frequency varies meaningfully across the user population.

The largest frequency bucket is 21–40 impressions (34.7%), followed by 41–60 impressions (20.1%).

More than half of exposed users received between 21 and 60 impressions, indicating substantial campaign exposure.

Approximately 10.7% of users received more than 60 impressions, creating an opportunity to investigate frequency saturation and whether incremental lift varies across exposure levels.

The distribution provides sufficient variation for future segmentation and lift analysis.

In [9]:
# ==========================================
# VALIDATE FACT_AD_EXPOSURES
# ==========================================

print("Rows:", len(fact_ad_exposures))
print("Unique users:", fact_ad_exposures["user_id"].nunique())
print("Duplicate user_ids:", fact_ad_exposures["user_id"].duplicated().sum())

print("Total impressions in raw:", raw_ad_events["impression_flag"].sum())
print("Total impressions in fact:", fact_ad_exposures["total_impressions"].sum())

print("Total clicks in raw:", raw_ad_events["click_flag"].sum())
print("Total clicks in fact:", fact_ad_exposures["total_clicks"].sum())

print("Total video completes in raw:", raw_ad_events["video_complete_flag"].sum())
print("Total video completes in fact:", fact_ad_exposures["total_video_completes"].sum())

Rows: 9718
Unique users: 9718
Duplicate user_ids: 0
Total impressions in raw: 341675
Total impressions in fact: 341675
Total clicks in raw: 4498
Total clicks in fact: 4498
Total video completes in raw: 127014
Total video completes in fact: 127014


### fact_ad_exposures Validation

The fact table contains 9,718 unique users and no duplicate user records.

Aggregate metrics were validated against the source data:


This confirms that the transformation successfully converted event-level data into a user-level exposure table without introducing duplication or data loss.

The table is now suitable for user-level incrementality analysis.

In [10]:
raw_sub_events = pd.read_csv(RAW + 'raw_subscription_events.csv')
print({f'raw_sub_event: {raw_sub_events.shape}'})

raw_sub_events.head()

{'raw_sub_event: (3650, 8)'}


,subscription_event_id,user_id,subscription_start_date,subscription_end_date,plan_type,plan_amount,status,event_created_date
0,SUB_00000001,U1017777,2021-08-01,NaN,yearly,87.85,trial,2021-08-03
1,SUB_00000002,U1019102,2021-08-03,2021-09-12,monthly,12.20,canceled,2021-08-03
2,SUB_00000003,U1005461,2021-08-03,NaN,yearly,116.06,active,2021-08-04
3,SUB_00000004,U1023954,2021-08-04,NaN,yearly,82.59,active,2021-08-05
4,SUB_00000005,U1020555,2021-08-04,NaN,quarterly,35.43,active,2021-08-05


In [11]:
# ==========================================
# BUILD FACT TABLE: fact_conversions
# ==========================================

# Ensure date is datetime
raw_sub_events["subscription_start_date"] = pd.to_datetime(
    raw_sub_events["subscription_start_date"]
)

# Each row represents a subscription, so every row is a conversion
fact_conversions = (
    raw_sub_events
    .copy()
    .sort_values(["user_id", "subscription_start_date"])
    .drop_duplicates(subset="user_id", keep="first")
    [
        [
            "user_id",
            "subscription_start_date",
            "plan_type",
            "plan_amount",
            "status"
        ]
    ]
    .rename(
        columns={
            "subscription_start_date": "first_conversion_date",
            "plan_amount": "conversion_amount",
            "status": "subscription_status"
        }
    )
)

fact_conversions["converted_flag"] = 1

# Reorder columns
fact_conversions = fact_conversions[
    [
        "user_id",
        "converted_flag",
        "first_conversion_date",
        "plan_type",
        "conversion_amount",
        "subscription_status"
    ]
]

print(fact_conversions.head())
print(fact_conversions.shape)

       user_id  converted_flag first_conversion_date  plan_type  \
1567  U1000004               1            2023-01-01  quarterly   
2538  U1000014               1            2023-09-18     yearly   
194   U1000023               1            2021-10-30  quarterly   
192   U1000041               1            2021-10-30     yearly   
2798  U1000073               1            2023-11-23    monthly   

      conversion_amount subscription_status  
1567              33.63            canceled  
2538             125.54            canceled  
194               34.37               trial  
192              126.58              active  
2798              13.84              active  
(3650, 6)


In [12]:
# Save fact tables as csv files 
fact_ad_exposures.to_csv(
    "../data/processed/fact_ad_exposures.csv",
    index=False
)

fact_conversions.to_csv(
    "../data/processed/fact_conversions.csv",
    index=False
)

In [13]:
print(fact_ad_exposures.shape)
print(fact_conversions.shape)

(9718, 16)
(3650, 6)


In [14]:
print("Rows:", len(fact_conversions))
print("Unique Users:", fact_conversions["user_id"].nunique())
print("Duplicate Users:", fact_conversions["user_id"].duplicated().sum())

overlap = len(
    set(dim_users["user_id"]) &
    set(fact_conversions["user_id"])
)

print("Users found in both dim_users and fact_conversions:", overlap)

Rows: 3650
Unique Users: 3650
Duplicate Users: 0


NameError: name 'dim_users' is not defined

In [ ]:
print("dim_users sample:")
print(dim_users["user_id"].head())
print(dim_users["user_id"].tail())

print("\nfact_conversions sample:")
print(fact_conversions["user_id"].head())
print(fact_conversions["user_id"].tail())

dim_users sample:
0    U1011592
1    U1008364
2    U1001620
3    U1004538
4    U1015135
Name: user_id, dtype: str
341670    U1024870
341671    U1022274
341672    U1024652
341673    U1002138
341674    U1018879
Name: user_id, dtype: str

fact_conversions sample:
1567    U1000004
2538    U1000014
194     U1000023
192     U1000041
2798    U1000073
Name: user_id, dtype: str
240     U1024977
837     U1024980
952     U1024983
2191    U1024984
1405    U1024999
Name: user_id, dtype: str


In [ ]:
print("dim_users:", dim_users.shape)
print("fact_ad_exposures:", fact_ad_exposures.shape)
print("fact_conversions:", fact_conversions.shape)

overlap = len(
    set(dim_users["user_id"])
    &
    set(fact_conversions["user_id"])
)

print("conversion overlap:", overlap)

dim_users: (25000, 16)
fact_ad_exposures: (9718, 16)
fact_conversions: (3650, 6)
conversion overlap: 3650
